# Notebook 01 - Attribution Basic (RQ1)

**RQ1:** Which social media channel contributes the most to sales conversions compared to other channels?

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd().resolve(), Path.cwd().resolve() / "notebooks", Path.cwd().resolve() / "analysis_python" / "notebooks", Path.cwd().resolve().parent / "notebooks"]:
    if (candidate / "notebook_header.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not find notebook_header.py")

import numpy as np
import pandas as pd

from notebook_header import OUTPUT_DIR as BASE_OUTPUT_DIR, RANDOM_SEED, load_journeys, load_touchpoints

OUTPUT_DIR = BASE_OUTPUT_DIR / "rq1_basic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 140)
pd.set_option("display.precision", 4)
np.random.seed(RANDOM_SEED)

## 1. Load Cleaned Data

The source files are `data_touchpoints.csv` and `data_journeys.csv` from the cleaned data folder. The precomputed fields `User_Converted`, `Is_First_Touch`, `Is_Last_Touch`, `N_Touchpoints`, and `Linear_Weight` are used as-is.

In [2]:
df_tp = load_touchpoints()
df_jr = load_journeys()

assert len(df_tp) == 10_000, f"Expected 10,000 touchpoints, got {len(df_tp)}"
assert df_tp["User ID"].nunique() == 2_847
assert len(df_jr) == 2_847
assert int(df_jr["Converted"].sum()) == 2_381
assert df_tp.isna().sum().sum() == 0
assert df_jr.isna().sum().sum() == 0

print(f"Touchpoints: {df_tp.shape}")
print(f"Journeys:    {df_jr.shape}")
print(f"Converted users: {int(df_jr['Converted'].sum())} / {len(df_jr)} ({df_jr['Converted'].mean()*100:.2f}%)")

Touchpoints: (10000, 13)
Journeys:    (2847, 9)
Converted users: 2381 / 2847 (83.63%)


## 2. First-Touch, Last-Touch, and Linear Attribution

In [3]:
converted_journeys = df_jr[df_jr["Converted"]]
converted_touchpoints = df_tp[df_tp["User_Converted"]]
converted_users = int(df_jr["Converted"].sum())

first_touch_share = (converted_journeys["First_Touch_Channel"].value_counts() / converted_users * 100).round(2)
last_touch_share = (converted_journeys["Last_Touch_Channel"].value_counts() / converted_users * 100).round(2)
linear_credit = converted_touchpoints.groupby("Channel")["Linear_Weight"].sum()
linear_share = (linear_credit / linear_credit.sum() * 100).round(2)

assert abs(linear_credit.sum() - converted_users) < 1e-6

attribution = pd.DataFrame({
    "first_touch_pct": first_touch_share,
    "last_touch_pct": last_touch_share,
    "linear_pct": linear_share,
}).fillna(0).sort_values("linear_pct", ascending=False)

for col in attribution.columns:
    assert abs(attribution[col].sum() - 100) < 0.1, f"{col} does not sum to 100%"

attribution

,first_touch_pct,last_touch_pct,linear_pct
Direct Traffic,17.26,17.85,17.15
Display Ads,17.98,16.84,17.10
Referral,17.14,16.13,16.75
Social Media,16.34,16.09,16.70
Email,15.71,16.51,16.28
Search Ads,15.58,16.59,16.02


## 3. Row-Level Conversion Rate by Channel

This is not an attribution model; it is a descriptive benchmark showing how often touchpoints in each channel have `Conversion = Yes`.

In [4]:
channel_conversion = df_tp.groupby("Channel").agg(
    touchpoints=("User ID", "count"),
    conversion_touchpoints=("Is_Conversion", "sum"),
)
channel_conversion["conversion_rate"] = channel_conversion["conversion_touchpoints"] / channel_conversion["touchpoints"]
channel_conversion["conversion_rate_pct"] = channel_conversion["conversion_rate"] * 100
channel_conversion = channel_conversion.sort_values("conversion_rate_pct", ascending=False)
channel_conversion

,touchpoints,conversion_touchpoints,conversion_rate,conversion_rate_pct
Channel,,,,
Email,1654,830,0.5018,50.1814
Referral,1685,841,0.4991,49.9110
Display Ads,1669,828,0.4961,49.6105
Direct Traffic,1721,853,0.4956,49.5642
Social Media,1662,820,0.4934,49.3381
Search Ads,1609,772,0.4798,47.9801


## 4. Social Media Position

In [5]:
social_media = attribution.loc["Social Media"]
rank_table = attribution.rank(ascending=False, method="min").astype(int)

rq1_summary = pd.DataFrame([
    {"metric": "Social Media first-touch share (%)", "value": social_media["first_touch_pct"]},
    {"metric": "Social Media last-touch share (%)", "value": social_media["last_touch_pct"]},
    {"metric": "Social Media linear share (%)", "value": social_media["linear_pct"]},
    {"metric": "Social Media first-touch rank", "value": int(rank_table.loc["Social Media", "first_touch_pct"])},
    {"metric": "Social Media last-touch rank", "value": int(rank_table.loc["Social Media", "last_touch_pct"])},
    {"metric": "Social Media linear rank", "value": int(rank_table.loc["Social Media", "linear_pct"])},
    {"metric": "Top first-touch channel", "value": attribution["first_touch_pct"].idxmax()},
    {"metric": "Top last-touch channel", "value": attribution["last_touch_pct"].idxmax()},
    {"metric": "Top linear channel", "value": attribution["linear_pct"].idxmax()},
])

rq1_summary

,metric,value
0,Social Media first-touch share (%),16.34
1,Social Media last-touch share (%),16.09
2,Social Media linear share (%),16.7
3,Social Media first-touch rank,4
4,Social Media last-touch rank,6
5,Social Media linear rank,4
6,Top first-touch channel,Display Ads
7,Top last-touch channel,Direct Traffic
8,Top linear channel,Direct Traffic


## 5. Save Paper-Ready Outputs

In [6]:
attribution.to_csv(OUTPUT_DIR / "01_attribution_share.csv")
channel_conversion.to_csv(OUTPUT_DIR / "01_channel_conversion_rate.csv")
rq1_summary.to_csv(OUTPUT_DIR / "01_social_media_summary.csv", index=False)

print("Saved:")
print(f"- {OUTPUT_DIR / '01_attribution_share.csv'}")
print(f"- {OUTPUT_DIR / '01_channel_conversion_rate.csv'}")
print(f"- {OUTPUT_DIR / '01_social_media_summary.csv'}")

Saved:
- F:\Sem 4\DAP391\Social-Media-Campaign-Attribution-for-E-commerce-Sales\analysis_python\outputs\rq1_basic\01_attribution_share.csv
- F:\Sem 4\DAP391\Social-Media-Campaign-Attribution-for-E-commerce-Sales\analysis_python\outputs\rq1_basic\01_channel_conversion_rate.csv
- F:\Sem 4\DAP391\Social-Media-Campaign-Attribution-for-E-commerce-Sales\analysis_python\outputs\rq1_basic\01_social_media_summary.csv


## 6. RQ1 conclusion


  The attribution results do not support the claim that Social Media is the strongest conversion channel in this dataset. Social Media receives 16.34% of First-Touch credit, 16.09% of Last-Touch credit, and 16.70% under Linear attribution. This places it 4th out of 6 channels under First-Touch, 6th under Last-Touch, and 4th under Linear. Social Media appears consistently in the conversion paths, but it does not lead under any of the three models.

  The leading channel depends on the attribution rule. Display Ads ranks first under First-Touch at 17.98%, while Direct Traffic ranks first under both Last-Touch (17.85%) and Linear attribution (17.15%). The row-level conversion rates show a similar pattern: Social Media is 49.34%, below Email (50.18%), Referral (49.91%), Display Ads (49.61%), and Direct Traffic (49.56%). These gaps are narrow, so they are better read as descriptive differences than as evidence that one channel clearly outperforms the others.

  Overall, the channel mix in this dataset is fairly balanced. Social Media contributes to conversions, but it does not show a clear edge over the other channels in any of the three attribution models. The more important finding is how evenly distributed the attribution shares are. That makes strong claims about any single channel difficult to justify from this data alone.
